In [ ]:
import ROOT
import math
# import pandas as pd

# print(f"ROOT version: {ROOT.__version__}")
import numpy as np
from scipy.special import j0
import plotly.graph_objects as go
from scipy.integrate import fixed_quad


In [ ]:
def read_data_file(filename):
    """Read data file and return arrays for x, y, y_error"""
    x_vals = []
    y_vals = []
    y_errs = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 3:
                    x_vals.append(float(parts[0]))
                    y_vals.append(float(parts[1]))
                    y_errs.append(float(parts[2]))
    
    return x_vals, y_vals, y_errs

In [ ]:
# Load experimental data for all energies from ATLAS
x_atlas_all, y_atlas_all, yerr_atlas_all = read_data_file('../../../data/ens_atlas_difc0_2.dat')

# Function to process data for each energy block
def process_data(x_data, y_data, yerr_data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        if end is None:
            end = len(x_data)
        x_values.append(x_data[start:end])
        y_values.append(y_data[start:end])
        y_errors.append(yerr_data[start:end])
    
    return x_values, y_values, y_errors


In [ ]:
import numpy as np
import ROOT

def process_data(x_all, y_all, yerr_all, blocks):
    """
    Process and split data into blocks (energy ranges).
    Handles ROOT buffers and converts to proper numpy arrays.
    
    Parameters:
        x_all: x-values (ROOT buffer or array-like)
        y_all: y-values (ROOT buffer or array-like)
        yerr_all: y-errors (ROOT buffer or array-like)
        blocks: list of tuples (start_idx, end_idx) for each energy range
        
    Returns:
        x_list, y_list, yerr_list: lists of numpy arrays, one per energy block
    """
    
    def safe_extract(data, start, end):
        """
        Safely extract data from ROOT buffers or array-like objects.
        """
        # Handle ROOT LowLevelView buffers
        if hasattr(data, '__len__') and hasattr(data, '__getitem__'):
            try:
                # Extract slice and convert to list first, then numpy
                if end is None:
                    extracted = [float(data[i]) for i in range(start, len(data))]
                else:
                    extracted = [float(data[i]) for i in range(start, end)]
                return np.array(extracted, dtype=np.float64)
            except Exception as e:
                print(f"Error in safe_extract: {e}")
                print(f"Data type: {type(data)}")
                print(f"Start: {start}, End: {end}")
                raise
        else:
            raise TypeError(f"Unsupported data type: {type(data)}")
    
    x_list = []
    y_list = []
    yerr_list = []
    
    for start, end in blocks:
        x_block = safe_extract(x_all, start, end)
        y_block = safe_extract(y_all, start, end)
        yerr_block = safe_extract(yerr_all, start, end)
        
        # Verify lengths match
        if not (len(x_block) == len(y_block) == len(yerr_block)):
            raise ValueError(f"Length mismatch in block ({start}, {end}): "
                           f"x={len(x_block)}, y={len(y_block)}, err={len(yerr_block)}")
        
        x_list.append(x_block)
        y_list.append(y_block)
        yerr_list.append(yerr_block)
    
    return x_list, y_list, yerr_list


# ============================================================================
# MAIN PROCESSING CODE
# ============================================================================

# ranges for each energy 
atlas_blocks = [(0, 29), (29, 58), (58, None)]

# Process data with improved handling
x_atlas, y_atlas, yerr_atlas = process_data(x_atlas_all, y_atlas_all, yerr_atlas_all, atlas_blocks)

# Extract values by energy - data is already numpy arrays from process_data
x_7_atlas = x_atlas[0]
y_7_atlas = y_atlas[0]
yerr_7_atlas = yerr_atlas[0]

x_8_atlas = x_atlas[1]
y_8_atlas = y_atlas[1]
yerr_8_atlas = yerr_atlas[1]

x_13_atlas = x_atlas[2]
y_13_atlas = y_atlas[2]
yerr_13_atlas = yerr_atlas[2]

# Verification
print("=== Data Extraction Summary ===")
print(f"√(s) = 7 TeV:")
print(f"  Type: {type(x_7_atlas)}")
print(f"  Shape: {x_7_atlas.shape}")
print(f"  Length: {len(x_7_atlas)} points")
print(f"  x range: [{x_7_atlas.min()}, {x_7_atlas.max()}]")

print(f"\n√(s) = 8 TeV:")
print(f"  Type: {type(x_8_atlas)}")
print(f"  Shape: {x_8_atlas.shape}")
print(f"  Length: {len(x_8_atlas)} points")
print(f"  x range: [{x_8_atlas.min()}, {x_8_atlas.max()}]")

print(f"\n√(s) = 13 TeV:")
print(f"  Type: {type(x_13_atlas)}")
print(f"  Shape: {x_13_atlas.shape}")
print(f"  Length: {len(x_13_atlas)} points")
print(f"  x range: [{x_13_atlas.min()}, {x_13_atlas.max()}]")

# print(x_7_atlas, x_8_atlas, x_13_atlas)



In [ ]:
# defining parameters/constants
b_0 = (33 - 6) / (12 * np.pi)
lambda_qcd = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25


#ensemble parameters
param_mg_atlas_pl = 0.421
param_eps_atlas_pl = 0.0753
param_a1_atlas_pl = 1.517
param_a2_atlas_pl = 2.05

In [ ]:
#--------------------------------------
# Eq 22 - GE
#--------------------------------------
def m2_pl(q2, mg):
    lambda2 = lambda_qcd ** 2
    rho_mg_2 = rho * (mg ** 2)
    ratio = math.log((q2 + rho_mg_2) / lambda2) / math.log(rho_mg_2 / lambda2)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

#--------------------------------------
# Eq 26 - GE
#--------------------------------------
def G_p(q2, a1, a2):
    t = -q2
    return np.exp(-(a1 * np.abs(t) + a2 * np.abs(t) ** 2))


#--------------------------------------
# Eq 24 - GE
#--------------------------------------
def alpha_D(q2, mg, m2_type):
    m2_func = m2_type(q2, mg)
    return 1.0 / (b_0 * (q2 + m2_func) * math.log((q2 + 4 * m2_func) / (lambda_qcd ** 2)))

#--------------------------------------
# Eq 7 - GE
#--------------------------------------
def T_1(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    G0 = G_p(q, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2


#--------------------------------------
# Eq 8 - GE
#--------------------------------------
def T_2(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


#--------------------------------------
# Eq 11 - GE
#--------------------------------------
def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

#--------------------------------------
# Eq 6 - GE
#--------------------------------------
def born_amp(diff_T, s, eps, t):
    alpha_pomeron = 1.0 + eps + 0.25 * t
    s_tilde = s/s0

    return 1j * s * 8 * (s_tilde**(alpha_pomeron - 1)) * diff_T

In [ ]:
def root_1d_integrator(func, lower_limit, upper_limit):
    
    """
    PURPOSE: Perform one-dimensional numerical integration using ROOT's 
    adaptive integration method.

    PARAMETERS:

        func (callable): Function to be integrated. Must accept a single 
            floating-point argument.

        lower_limit (float): Lower bound of the integration interval.

        upper_limit (float): Upper bound of the integration interval.

    RETURNS:
        tuple: Tuple containing:
            - [0] (float): Estimated value of the integral.
            - [1] (float): Estimated uncertainty of the integral.
    """

    # creating Functor to be used by ROOT's integrator
    functor = ROOT.Math.Functor1D(func)

    type = ROOT.Math.IntegrationOneDim.kADAPTIVE   # integration type
    absTol = 1e-4
    relTol = 1e-4
    size   = 20
    rule   = ROOT.Math.Integration.kGAUSS15   
        
    # defining Integrator object
    integrator = ROOT.Math.IntegratorOneDim(type, absTol, relTol, size, rule)
    integrator.SetFunction(functor)
    
    # calculating integral and error 
    result = integrator.Integral(lower_limit, upper_limit)
    error = integrator.Error()
    
    return result, error


In [ ]:
#--------------------------------------
# Eq 7 and Eq 8 - GE
#--------------------------------------

def k_integral(k, mg, a1, a2, m2_func, q):
    """
    PURPOSE: Define the integrand function over φ (phi) for a fixed k value.

    PARAMETERS:

        k (float): Momentum magnitude variable for the outer integral.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

    RETURNS:
        callable: Integrand function of φ (phi), defined as:
            f(φ) = k * [T₁(k, φ, mg, a1, a2, m2_func, q) - T₂(k, φ, mg, a1, a2, m2_func, q)].
    """
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    return integrand


def phi_integral(phi, mg, a1, a2, m2_func, q, k_max):
    """
    PURPOSE: Compute the inner integral over k for a fixed φ (phi) value 
    using ROOT's 1D integrator.

    PARAMETERS:

        phi (float): Azimuthal angle variable (in radians).

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

        k_max (float): Upper limit for the k integration.

    RETURNS:
        float: Estimated value of the k-integral for the given φ (phi):
            ∫₀^{k_max} k [T₁(k, φ) - T₂(k, φ)] dk.
    """
    def inner_in_k(k):
        return k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                    T_2(k, phi, mg, a1, a2, m2_func, q))
    
    result, _ = root_1d_integrator(inner_in_k, 0, k_max)
    return result


def compute_k_phi_integral(mg, a1, a2, m2_func, q, k_max):
    """
    PURPOSE: Compute the full two-dimensional integral over k and φ (phi),
    corresponding to Eqs. (7) and (8) in the GE model.

    PARAMETERS:

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        q (float): Momentum transfer variable in the scattering process.

        k_max (float): Upper limit for the k integration.

    RETURNS:
        tuple: Tuple containing:
            - [0] (float): Estimated value of the double integral:
                ∫₀^{2π} ∫₀^{k_max} k [T₁(k, φ) - T₂(k, φ)] dk dφ
            - [1] (float): Estimated uncertainty from the outer φ integration.
    """
    result, error = root_1d_integrator(
        lambda phi: phi_integral(phi, mg, a1, a2, m2_func, q, k_max),
        0,
        2 * math.pi
    )
    return result, error



# TESTING 

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0
k_max = 13000

compute_k_phi_integral(mg, a1, a2, m2_pl, q, k_max)

In [ ]:
#--------------------------------------  
#   COMPUTING SIGMA TOT BORN
#--------------------------------------


#start parameters
start_sqrt_s = 100
max_sqrt_s = 13000
step_size = 500

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0

lst_sigma_tot_born_list = []
lst_sqrt_s_list = []

# start initial sqrt for while loop  
current_sqrt_s = start_sqrt_s

# while current_sqrt_s <= max_sqrt_s:

#     current_s = current_sqrt_s**2

#     # calculates diff t (eq 7 and 8)
#     diff_t,_ = compute_k_phi_integral(mg, a1, a2, m2_pl, q, current_sqrt_s)

#     # calculating born amplitude
#     born_amp_value = born_amp(diff_t, current_s, param_eps_atlas_pl, 0)
#     # print(born_amp_value)

#     #calculating sigma tot born
#     born_sigma_tot_value = born_sigma_tot(born_amp_value, current_s)
#     # print(born_sigma_tot_value)

#     # append results to list
#     lst_sigma_tot_born_list.append(born_sigma_tot_value)
#     lst_sqrt_s_list.append(current_sqrt_s)

#     # increase step
#     current_sqrt_s += step_size 
#     print(born_sigma_tot_value)

In [ ]:
#--------------------------------------
# Eq 23 - EIK
#--------------------------------------


def chi_eikonal(s, b, eps, mg, a1, a2, m2_func, born_amp_func):
    """
    PURPOSE: Compute the complex eikonal function χ(s, b) as defined in Eq. (23),
    using the Born amplitude and the nested integral over k and φ.

    PARAMETERS:

        s (float): Mandelstam variable s (squared center-of-mass energy).

        b (float): Impact parameter in femtometers (fm) or GeV⁻¹, depending on model units.

        eps (float): Model parameter ε controlling energy dependence of the amplitude.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer q.

        born_amp_func (callable): Function that computes the complex Born amplitude:
            A_Born(diff_T, s, eps, t), where t = -q² and diff_T is obtained from the 
            nested integral over k and φ.

    RETURNS:
        complex: Complex value of the eikonal function χ(s, b), computed as:
            (1 / s) × [∫ Re(A_Born) dq  +  i ∫ Im(A_Born) dq].

    NOTES:

        - Implements Eq. (23) from the generalized eikonal (GE) formalism.
        - Uses `root_1d_integrator` for numerical evaluation of the q-integral.
        - The integration limits (0 → 0.2) are set empirically and may depend 
          on the energy range or chosen model normalization.
    """

    def integrand_real(q):
        t = -q**2  
        
        # Compute diff_T for this q value
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)
        )

        # Compute Born amplitude
        amp_born = born_amp_func(diff_T, s, eps, t)
        
        # Integrand for the real part
        return q * j0(b * q) * amp_born.real
    
    def integrand_imag(q):
        t = -q**2
        
        # Compute diff_T for this q value
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)
        )

        # Compute Born amplitude
        amp_born = born_amp_func(diff_T, s, eps, t)
        
        # Integrand for the imaginary part
        return q * j0(b * q) * amp_born.imag
    
    # Perform numerical integration for real and imaginary components
    real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, 0.2)
    imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, 0.2)

    # Combine real and imaginary parts
    integral_result = real_integral_result + 1j * imag_integral_result

    return integral_result / s

current_sqrt_s = 10000
# while current_sqrt_s <= max_sqrt_s+1:
#     current_s = current_sqrt_s**2
#     print(current_sqrt_s)
#     print(chi_eikonal(current_s, 0.1, param_eps_atlas_pl, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl,born_amp))
#     current_sqrt_s += 200

In [ ]:
#--------------------------------------
# Eq 24 - EIK (using the new chi_eikonal)
#--------------------------------------

def eik_amp(s, eps, mg, a1, a2, m2_func, born_amp_func, q_max=0.2):
    """
    PURPOSE: Compute the eikonalized scattering amplitude A_eik(s, t) 
    as defined in Eq. (24) using the eikonal function χ(s, b).

    PARAMETERS:

        s (float): Mandelstam variable s (squared center-of-mass energy).

        t (float): Mandelstam variable t (momentum transfer squared, typically negative).

        eps (float): Model parameter ε controlling the energy dependence of the amplitude.

        mg (float): Effective gluon mass parameter.

        a1 (float): Model parameter a₁ related to the form factor.

        a2 (float): Model parameter a₂ related to the form factor.

        m2_func (callable): Function returning the squared mass term m²(q²) as a function 
            of momentum transfer.

        born_amp_func (callable): Function that computes the complex Born amplitude:
            A_Born(diff_T, s, eps, t), where diff_T is obtained from the nested integral 
            over k and φ.

        q_max (float, optional): Maximum momentum transfer q used internally in χ(s, b) 
            integration. Default is 0.2.

    RETURNS:
        complex: Complex eikonalized amplitude A_eik(s, t), given by:
            i s × ∫₀^{b_max} b J₀(b√(-t)) [1 - exp(iχ(s, b))] db.

    NOTES:

        - Implements Eq. (24) from the generalized eikonal (GE) formalism.
        - Uses the `chi_eikonal` function to evaluate χ(s, b) at each b value.
        - The integration is performed over b ∈ [0, 30], which may be adjusted 
          depending on the physical range or model normalization.
        - `root_1d_integrator` is used for adaptive numerical integration.
    """

    # q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
    def integrand_real(b_val):
        # Compute eikonal function χ(s, b)
        chi_val = chi_eikonal(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func)
        
        # Compute [1 - exp(iχ(s, b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s, b))]
        return (b_val * j0(b_val * 0) * one_minus_exp).real
    
    def integrand_imag(b_val):
        # Compute eikonal function χ(s, b)
        chi_val = chi_eikonal(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func)
        
        # Compute [1 - exp(iχ(s, b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s, b))]
        return (b_val * j0(b_val * 0) * one_minus_exp).imag
    
    # Integrate real and imaginary parts separately
    real_integral, _ = root_1d_integrator(integrand_real, 0.0, 30)
    imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, 30)
    
    # Combine real and imaginary components
    integral_result = real_integral + 1j * imag_integral
    
    # Final amplitude: i s times the integral
    return 1j * s * integral_result

# print(eik_amp(13000**2, param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl, born_amp))

# current_sqrt_s = 10000
# while current_sqrt_s <= max_sqrt_s+1:
#     current_s = current_sqrt_s**2
#     print(current_sqrt_s)
#     print(eik_amp(current_s, param_eps_atlas_p4l, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl, born_amp))
#     current_sqrt_s += 200


In [ ]:
import ROOT
import numpy as np
import math


def root_minimize(func,
                  ndim,
                  minimizerName="Minuit2",
                  algoName="",
                  mg_init=None,\
                  eps_init=None,
                  a1_init=None,
                  a2_init=None,
                  stepSize=None,
                  maxFunctionCalls=1000000,
                  maxIterations=10000,
                  tolerance=1e-8,
                  printLevel=0):
    """
    PURPOSE: Generic wrapper to perform parameter minimization using ROOT's 
    built-in minimizers, returning parameter values, Hesse and MINOS errors.

    PARAMETERS:

        func (callable): Function to minimize. Should accept a list or numpy array 
            of length `ndim`.

        ndim (int): Number of parameters to minimize.

        minimizerName (str, optional): Minimizer to use (Minuit, Minuit2, 
            GSLMultiMin, GSLSimAn, Genetic, etc.). Default is "Minuit2".

        algoName (str, optional): Specific algorithm to use (Migrad, BFGS, 
            ConjugateFR, Simplex, etc.). Default is "".

        mg_init (float or None, optional): Initial guess for parameter 'mg'. 
            Defaults to 0.0 if None.

        eps_init (float or None, optional): Initial guess for parameter 'eps'. 
            Defaults to 0.0 if None.

        a1_init (float or None, optional): Initial guess for parameter 'a1'. 
            Defaults to 0.0 if None.

        a2_init (float or None, optional): Initial guess for parameter 'a2'. 
            Defaults to 0.0 if None.

        stepSize (list of floats or None, optional): Step sizes for each parameter. 
            Defaults to 0.01 for all parameters.

        maxFunctionCalls (int, optional): Maximum allowed function evaluations. 
            Default is 1,000,000.

        maxIterations (int, optional): Maximum allowed iterations. Default is 10,000.

        tolerance (float, optional): Desired convergence tolerance. Default is 1e-8.

        printLevel (int, optional): Verbosity of the minimizer (0=quiet, 1=normal, 
            2=verbose). Default is 1.

    RETURNS:
        dict: Dictionary containing minimization results:
            - 'success' (bool): Whether minimization converged successfully.
            - 'x' (numpy.ndarray): Parameter values at minimum.
            - 'status' (int): Minimizer status (0 = success).
            - 'hesse_errors' (numpy.ndarray): Symmetric Hesse errors.
            - 'minos_errors_low' (numpy.ndarray): Lower MINOS errors.
            - 'minos_errors_up' (numpy.ndarray): Upper MINOS errors.
    """
    
    #-------------------
    #  SET STARTING POINT
    #-------------------

    param_names = ["mg", "eps", "a1", "a2"]

    startPoint = [
        mg_init if mg_init is not None else 0.0,
        eps_init if eps_init is not None else 0.0,
        a1_init if a1_init is not None else 0.0,
        a2_init if a2_init is not None else 0.0
    ][:ndim]  # garante que só use ndim parâmetros

    # OLD VERSION OF THE CODE ABOVE
    # init_map = [mg_init, eps_init, a1_init, a2_init]
    # startPoint = []
    # for i in range(ndim):
    #     if init_map[i] is not None:
    #         startPoint.append(init_map[i])
    #     else:
    #         startPoint.append(0.0)  # fallback default

    
    # --------------------------------------------------------------

    #-------------------
    #  SET STEP SIZE
    #-------------------
    if stepSize is None:
        stepSize = [0.01] * ndim

    #-------------------
    #  SET CONFIDENCE LEVEL FOR 4D CASE 90% CL
    #-------------------
    errordef = 7.78
    
    #-------------------
    #  CREATE MINIMIZER
    #-------------------

    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if not minimizer:
        raise RuntimeError(f"Cannot create minimizer \"{minimizerName}\"")
    

    #-------------------
    #  SET OPTIONS
    #-------------------

    minimizer.SetMaxFunctionCalls(maxFunctionCalls)
    minimizer.SetMaxIterations(maxIterations)
    minimizer.SetTolerance(tolerance)
    minimizer.SetPrintLevel(printLevel)
    minimizer.SetErrorDef(errordef)
    f = ROOT.Math.Functor(func, ndim)
    minimizer.SetFunction(f)

    
    variable = list(startPoint)

    #-------------------
    #  SET PARAMETERS
    #-------------------

    # renaming variable names to match model parameters and set parameters 
    if ndim >= 1:
        minimizer.SetVariable(0, param_names[0], variable[0], stepSize[0])
    if ndim >= 2:
        minimizer.SetVariable(1, param_names[1], variable[1], stepSize[1])
    if ndim >= 3:
        minimizer.SetVariable(2, param_names[2], variable[2], stepSize[2])
    if ndim >= 4:
        minimizer.SetVariable(3, param_names[3], variable[3], stepSize[3])

    # OLD VERSION OF THE CODE ABOVE
    # for i in range(ndim):
    #     if i < len(param_names):
    #         name = param_names[i]
    #     else:
    #         name = f"x{i}"
    #     minimizer.SetVariable(i, name, variable[i], stepSize[i])

    

    
    """
    could replace the code above by the following code to set parameters without renaming
    for i in range(ndim):
        minimizer.SetVariable(i, f"x{i}", variable[i], stepSize[i])
    """

    #-------------------
    #  RUN MINIMIZATION
    #-------------------

    minimization = minimizer.Minimize()
    if not minimization:
        return {'success': False}
    

    #-------------------
    # GET HESSE ERROR
    #-------------------

    # Create empty arrays to store the results
    xs = np.zeros(ndim)           # parameter values at minimum
    hesse_errors = np.zeros(ndim) # symmetric Hesse errors

    # Loop over each parameter and extract the value and Hesse error
    if ndim >= 1:
        xs[0] = minimizer.X()[0]
        hesse_errors[0] = minimizer.Errors()[0]
    if ndim >= 2:
        xs[1] = minimizer.X()[1]
        hesse_errors[1] = minimizer.Errors()[1]
    if ndim >= 3:
        xs[2] = minimizer.X()[2]
        hesse_errors[2] = minimizer.Errors()[2]
    if ndim >= 4:
        xs[3] = minimizer.X()[3]
        hesse_errors[3] = minimizer.Errors()[3]

    # OLD VERSION OF THE CODE ABOVE
    # for i in range(ndim):
    #     xs[i] = minimizer.X()[i]          # get the fitted value of parameter i
    #     hesse_errors[i] = minimizer.Errors()[i]  # get the Hesse error for parameter i


    #-------------------
    # GET MINOS ERROR
    #-------------------
    
    # Initialize arrays to store MINOS errors
    minos_errors_low = np.zeros(ndim)
    minos_errors_up = np.zeros(ndim)

    # Temporary arrays for ROOT's GetMinosError
    errLow = np.zeros(1, dtype=np.float64)
    errUp  = np.zeros(1, dtype=np.float64)

    if ndim >= 1:
        success = minimizer.GetMinosError(0, errLow, errUp)
        if success:
            minos_errors_low[0] = errLow[0]
            minos_errors_up[0] = errUp[0]
        else:
            minos_errors_low[0] = -hesse_errors[0]
            minos_errors_up[0] = hesse_errors[0]

    if ndim >= 2:
        success = minimizer.GetMinosError(1, errLow, errUp)
        if success:
            minos_errors_low[1] = errLow[0]
            minos_errors_up[1] = errUp[0]
        else:
            minos_errors_low[1] = -hesse_errors[1]
            minos_errors_up[1] = hesse_errors[1]

    if ndim >= 3:
        success = minimizer.GetMinosError(2, errLow, errUp)
        if success:
            minos_errors_low[2] = errLow[0]
            minos_errors_up[2] = errUp[0]
        else:
            minos_errors_low[2] = -hesse_errors[2]
            minos_errors_up[2] = hesse_errors[2]

    if ndim >= 4:
        success = minimizer.GetMinosError(3, errLow, errUp)
        if success:
            minos_errors_low[3] = errLow[0]
            minos_errors_up[3] = errUp[0]
        else:
            minos_errors_low[3] = -hesse_errors[3]
            minos_errors_up[3] = hesse_errors[3]

    # OLD VERSION OF THE CODE ABOVE
    # for i in range(ndim):
    #     success = minimizer.GetMinosError(i, errLow, errUp)
    #     if success:
    #         minos_errors_low[i] = errLow[0]
    #         minos_errors_up[i] = errUp[0]
    #     else:
    #         # fallback to Hesse errors if MINOS fails
    #         minos_errors_low[i] = -hesse_errors[i]
    #         minos_errors_up[i] = hesse_errors[i]



    # print results
    print("\nMinimization results (values ± Hesse ± MINOS):")
    for i in range(ndim):
        print(f"{param_names[i]}: {xs[i]:.6f} "
              f"± {hesse_errors[i]:.6f} "
              f"[{minos_errors_low[i]:+.6f}, {minos_errors_up[i]:+.6f}]")

    print(f"\nStatus: {minimizer.Status()} (0 = success)\n")
    # ----------------------

    return {
        'success': minimization and minimizer.Status() == 0,
        'x': xs,
        'status': minimizer.Status(),
        'hesse_errors': hesse_errors,
        'minos_errors_low': minos_errors_low,
        'minos_errors_up': minos_errors_up,
    }


In [ ]:
import ROOT
import numpy as np

def get_chi2_minimization(func_model, x_data, y_data, y_errors, initial_params, param_limits, 
             xmin, xmax, root_minimize, minimizerName="Minuit2", algoName="Migrad", 
             **minimize_options):
    """
    PURPOSE: Function to perform chi-squared minimization using ROOT's built-in methods.

    PARAMETERS:
        func_model (callable): Model function to fit. Should have signature:
                              func_model(x, params) where params is dict or array
        x_data (array): x-values of the data points.
        y_data (array): y-values of the data points.
        y_errors (array): uncertainties on y-values.
        initial_params (array): initial guess for the parameters.
        param_limits (list of tuples, optional): parameter limits [(min, max), ...].
        xmin (float): minimum x-value for the fit range.
        xmax (float): maximum x-value for the fit range.
        root_minimize (function): ROOT's minimization function.
        minimizerName (str, optional): minimizer to use (default is "Minuit2").
        algoName (str, optional): minimization algorithm (default is "Migrad").
        **minimize_options: additional options for root_minimize.
    
    RETURNS:
        dict: best-fit parameters and fit results.
    """

    # Number of data points
    n_points = len(x_data)
    
    # parameter names for output
    param_names = ['mg', 'eps', 'a1', 'a2']

    # number of parameters
    npar = len(initial_params)
    
    # Convert data to numpy arrays to ensure consistency
    x_data = np.array(x_data, dtype=np.float64)
    y_data = np.array(y_data, dtype=np.float64)
    y_errors = np.array(y_errors, dtype=np.float64)
    
    # Define chi-square function
    def chi2_func(params):
        # Extract parameters as individual floats
        # This is CRITICAL - ROOT passes a buffer-like object
        param_array = np.zeros(npar, dtype=np.float64)
        for i in range(npar):
            param_array[i] = float(params[i])
        
        chi2 = 0.0
        
        try:
            for i in range(n_points):
                # Calculate prediction from your model
                # Pass individual x value and parameter array
                y_pred = func_model(float(x_data[i]), param_array)
                
                # Handle complex return values
                if isinstance(y_pred, complex):
                    y_pred = abs(y_pred)**2
                else:
                    y_pred = float(y_pred)
                
                # Chi-squared contribution
                residual = (y_pred - float(y_data[i])) / float(y_errors[i])
                chi2 += residual**2
                
        except Exception as e:
            print(f"Error in chi2_func: {e}")
            print(f"params: {param_array}")
            return 1e10  # Return large chi2 on error
        
        return float(chi2)
    
    # get root_minimize arguments
    minimize_args = {
        'func': chi2_func,
        'ndim': npar,
        'minimizerName': minimizerName,
        'algoName': algoName,
        'mg_init': float(initial_params[0]),
        'eps_init': float(initial_params[1]),
        'a1_init': float(initial_params[2]),
        'a2_init': float(initial_params[3]),
    }
    minimize_args.update(minimize_options)
    
    # execute minimization
    result = root_minimize(**minimize_args)
    
    if result['success']:
        # calculate final chi-square with best-fit parameters
        chi2 = chi2_func(result['x'])
        ndf = n_points - npar
        
        # output dictionary
        output = {
            'chi2': chi2,
            'ndf': ndf,
            'chi2_dof': chi2 / ndf if ndf > 0 else 0.0,
            'parameters': result['x'].tolist(),
            'errors': result['hesse_errors'].tolist(),
            'minos_errors_low': result['minos_errors_low'].tolist(),
            'minos_errors_up': result['minos_errors_up'].tolist(),
            'param_names': param_names,
            'valid': True,
            'status': result['status'],
            'covariance_matrix': result.get('covariance_matrix', None)
        }

    else:
        output = {
            'chi2': 0.0,
            'ndf': 0,
            'chi2_dof': 0.0,
            'parameters': initial_params,
            'errors': [0.0] * npar,
            'minos_errors_low': [0.0] * npar,
            'minos_errors_up': [0.0] * npar,
            'param_names': param_names,
            'valid': False,
            'status': -1,
            'covariance_matrix': None
        }
    
    return output

In [ ]:
def born_dif_sigma(born_amp, s):
    return (born_amp * born_amp.imag())/(16 * np.pi * s) * 0.389379323

def eik_dif_sigma(eik_amp, s):
    return np.pi/s**2 * (eik_amp* eik_amp.imag()) * 0.389379323

In [ ]:
down = 0.5
up = 1.5



result = get_chi2_minimization(
    eik_amp(s = 13000**2, mg=param_mg_atlas_pl, eps=param_eps_atlas_pl, a1=param_a1_atlas_pl, a2=param_a2_atlas_pl, m2_func=m2_pl, born_amp_func=born_amp),
    x_7_atlas,
    y_7_atlas,
    yerr_7_atlas,
    [param_mg_atlas_pl, param_eps_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl],
    param_limits = [(down * param_mg_atlas_pl, up * param_mg_atlas_pl),
                    (down * param_eps_atlas_pl, up * param_eps_atlas_pl),
                    (down * param_a1_atlas_pl, up * param_a1_atlas_pl),
                    (down * param_a2_atlas_pl, up * param_a2_atlas_pl)],
    xmin=min(x_7_atlas),
    xmax=max(x_7_atlas),
    root_minimize=root_minimize,
    minimizerName='Minuit2',
    algoName='Migrad')

In [ ]:
import ROOT
import numpy as np
from scipy import stats

# Simple example: Fit data to integral of exponential decay
# Model: y(x) = integral from 0 to x of [A * exp(-lambda * t)] dt
#             = A/lambda * (1 - exp(-lambda * x))

# Step 1: Generate synthetic data
print("Generating synthetic data...")
np.random.seed(42)  # For reproducibility
x_data = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0]
# True parameters: A=3.0, lambda=0.8
A_true = 3.0
lambda_true = 0.8
y_data = [A_true/lambda_true * (1 - np.exp(-lambda_true*x)) for x in x_data]
# Add some noise
y_data = [y + np.random.normal(0, 0.2) for y in y_data]
y_errors = [0.2] * len(x_data)

print(f"Data points: {len(x_data)}")
print(f"True parameters: A={A_true}, lambda={lambda_true}")


# Step 2: Define chi2 function
def calculate_chi2(params):
    """Calculate chi2 for given parameters"""
    A = params[0]
    lam = params[1]
    
    # Avoid numerical issues
    if A <= 0 or lam <= 0:
        return 1e10
    
    # Create function to integrate
    func = ROOT.TF1("f", f"{A}*exp(-{lam}*x)", 0, 10)
    
    chi2 = 0.0
    for i in range(len(x_data)):
        # Model prediction: integral from 0 to x
        predicted = func.Integral(0, x_data[i])
        
        # Chi2 contribution
        residual = (y_data[i] - predicted) / y_errors[i]
        chi2 += residual**2
    
    return chi2


# Step 3: Setup minimizer
print("\nSetting up minimizer...")
n_params = 2
chi2_functor = ROOT.Math.Functor(calculate_chi2, n_params)

minimizer = ROOT.Math.Factory.CreateMinimizer("Minuit2", "Migrad")
minimizer.SetFunction(chi2_functor)

# Configuration
minimizer.SetMaxFunctionCalls(10000000)
minimizer.SetTolerance(1e-4)         # Convergence tolerance
minimizer.SetPrecision(1e-10)        # Function precision
minimizer.SetPrintLevel(0)           # Show minimization progress
minimizer.SetStrategy(2)             # 0=fast, 1=default, 2=careful

# ============================================================================
# SET 90% CONFIDENCE LEVEL
# ============================================================================
# For chi-square with 1 parameter at 90% CL: up = 2.71
# For chi-square with 2 parameters at 90% CL: up = 4.61
up_90_1param = stats.chi2.ppf(0.90, df=1)  # 2.7055
up_90_2param = stats.chi2.ppf(0.90, df=2)  # 4.6052

# Use 1-parameter value for individual parameter errors
minimizer.SetErrorDef(up_90_1param)
print(f"\nErrorDef set to {up_90_1param:.4f} for 90% CL (1 parameter)")
print(f"Note: For 2D contours, use {up_90_2param:.4f}")
# ============================================================================

# Set initial parameter values
minimizer.SetVariable(0, "A", 2.0, 0.1)           # amplitude
minimizer.SetVariable(1, "lambda", 0.5, 0.01)     # decay constant

# Optional: set reasonable limits
minimizer.SetVariableLimits(0, 0.1, 10.0)
minimizer.SetVariableLimits(1, 0.01, 5.0)


# Step 4: Minimize
print("\nMinimizing...")
success = minimizer.Minimize()

# Calculate Hessian matrix for error analysis
print("\nCalculating error matrix (Hesse)...")
minimizer.Hesse()


# Step 5: Display results
print("\n" + "="*60)
if success:
    print("MINIMIZATION SUCCESSFUL!")
else:
    print("MINIMIZATION FAILED!")
print("="*60)

chi2_min = minimizer.MinValue()
n_dof = len(x_data) - n_params

print(f"\nChi2_min = {chi2_min:.4f}")
print(f"NDF = {n_dof}")
print(f"Chi2/NDF = {chi2_min/n_dof:.4f}")
print(f"Status = {minimizer.Status()}")

print(f"\n{'='*60}")
print(f"FITTED PARAMETERS (90% Confidence Level)")
print(f"{'='*60}")

A_fit = minimizer.X()[0]
A_err = minimizer.Errors()[0]
lambda_fit = minimizer.X()[1]
lambda_err = minimizer.Errors()[1]

print(f"  A      = {A_fit:.4f} ± {A_err:.4f}  (true: {A_true})")
print(f"  lambda = {lambda_fit:.4f} ± {lambda_err:.4f}  (true: {lambda_true})")

print(f"\nCorrelation(A, lambda) = {minimizer.Correlation(0, 1):.4f}")


# Step 6: Get MINOS asymmetric errors at 90% CL (optional but recommended)
print(f"\n{'='*60}")
print("MINOS ASYMMETRIC ERRORS (90% CL)")
print(f"{'='*60}")

# Need to use ctypes for error parameters in PyROOT
import ctypes
err_low = ctypes.c_double()
err_up = ctypes.c_double()

for i in range(n_params):
    param_name = minimizer.VariableName(i)
    param_value = minimizer.X()[i]
    
    # Get MINOS errors (this respects the ErrorDef setting)
    minos_success = minimizer.GetMinosError(i, err_low, err_up)
    
    if minos_success:
        print(f"  {param_name:8s} = {param_value:8.4f} +{err_up.value:7.4f} -{abs(err_low.value):7.4f}")
    else:
        print(f"  {param_name:8s} = {param_value:8.4f} (MINOS failed)")


# Step 7: Summary table
print(f"\n{'='*60}")
print("SUMMARY TABLE")
print(f"{'='*60}")
print(f"{'Parameter':<12s} {'True':>10s} {'Fitted':>10s} {'Error (90% CL)':>18s}")
print(f"{'-'*60}")
print(f"{'A':<12s} {A_true:>10.4f} {A_fit:>10.4f} {A_err:>10.4f} (±)")
print(f"{'lambda':<12s} {lambda_true:>10.4f} {lambda_fit:>10.4f} {lambda_err:>10.4f} (±)")


# Step 8: Comparison of confidence levels
print(f"\n{'='*60}")
print("ERROR COMPARISON AT DIFFERENT CONFIDENCE LEVELS")
print(f"{'='*60}")

confidence_levels = {
    "68.27% (1σ)": 1.00,
    "90%": up_90_1param,
    "95%": stats.chi2.ppf(0.95, df=1),
    "95.45% (2σ)": 4.00,
    "99%": stats.chi2.ppf(0.99, df=1)
}

print(f"\nParameter: A")
print(f"{'CL':<15s} {'ErrorDef':>10s} {'Error':>10s}")
print(f"{'-'*40}")
for cl_name, up_value in confidence_levels.items():
    # Approximate scaling: error scales with sqrt(up)
    scaled_error = A_err * np.sqrt(up_value / up_90_1param)
    print(f"{cl_name:<15s} {up_value:>10.4f} {scaled_error:>10.4f}")

print(f"\nParameter: lambda")
print(f"{'CL':<15s} {'ErrorDef':>10s} {'Error':>10s}")
print(f"{'-'*40}")
for cl_name, up_value in confidence_levels.items():
    scaled_error = lambda_err * np.sqrt(up_value / up_90_1param)
    print(f"{cl_name:<15s} {up_value:>10.4f} {scaled_error:>10.4f}")



In [ ]:
#  \file
#  \ingroup tutorial_fit
#  \notebook -nodraw
#  Example on how to use the new Minimizer class in ROOT
#   Show usage with all the possible minimizers.
#  Minimize the Rosenbrock function (a 2D -function)
#
#  input : minimizer name + algorithm name
#  randomSeed: = <0 : fixed value: 0 random with seed 0; >0 random with given seed
#
#  \macro_code
#
#  \author Lorenzo Moneta

import ROOT
import numpy as np


def RosenBrock(vecx):
    x = vecx[0]
    y = vecx[1]
    return (y - x**2)**2 + (1 - x)**2

# New function: Example chi2 calculation for a linear fit
def LinearChi2(params, x_data, y_data, y_errors):
    """
    Calculate chi2 for a linear model y = a + b*x
    params[0] = intercept (a)
    params[1] = slope (b)
    """
    a = params[0]
    b = params[1]
    chi2 = 0.0
    for i in range(len(x_data)):
        y_pred = a + b * x_data[i]
        residual = y_data[i] - y_pred
        chi2 += (residual / y_errors[i])**2
    return chi2

def CalculateChi2PerDof(minimizer, n_data_points, n_parameters):
    """
    Calculate chi2 per degree of freedom
    chi2/dof = chi2 / (n_data_points - n_parameters)
    """
    chi2 = minimizer.MinValue()
    dof = n_data_points - n_parameters
    if dof <= 0:
        print("Warning: Degrees of freedom <= 0!")
        return chi2, dof, -1
    
    chi2_per_dof = chi2 / dof
    return chi2, dof, chi2_per_dof

# create minimizer giving a name and a name (optionally) for the specific algorithm
#  possible choices are:
#     minimizerName                  algoName
#
#     Minuit                     Migrad, Simplex,Combined,Scan  (default is Migrad)
#     Minuit2                    Migrad, BFGS, Simplex,Combined,Scan  (default is Migrad)
#     GSLMultiMin                ConjugateFR, ConjugatePR, BFGS, BFGS2, SteepestDescent
#     GSLSimAn
#     Genetic


def NumericalMinimization(minimizerName="Minuit2",
                          algoName="",
                          randomSeed=-1):

    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if (not minimizer):
        raise RuntimeError(
            "Cannot create minimizer \"{}\". Maybe the required library was not built?".format(minimizerName))

    # Set tolerance and other minimizer parameters, one can also use default
    # values

    minimizer.SetMaxFunctionCalls(1000000)  # working for Minuit/Minuit2
    # for GSL minimizers - no effect in Minuit/Minuit2
    minimizer.SetMaxIterations(10000)
    minimizer.SetTolerance(0.001)
    minimizer.SetPrintLevel(1)

    # Create function wrapper for minimizer

    f = ROOT.Math.Functor(RosenBrock, 2)

    # Evaluate function at a point
    x0 = np.array([-1., 2.])
    print("f(-1,1.2) = ", f(x0))

    # Starting point
    variable = [-1., 1.2]
    step = [0.01, 0.01]
    if (randomSeed >= 0):
        r = ROOT.TRandom2(randomSeed)
        variable[0] = r.Uniform(-20, 20)
        variable[1] = r.Uniform(-20, 20)

    minimizer.SetFunction(f)

    # Set the free variables to be minimized !
    minimizer.SetVariable(0, "x", variable[0], step[0])
    minimizer.SetVariable(1, "y", variable[1], step[1])

    # Do the minimization
    ret = minimizer.Minimize()

    xs = minimizer.X()
    print("Minimum: f({} , {}) = {}".format(xs[0],xs[1],minimizer.MinValue()))

    # Calculate chi2/dof for demonstration
    # For Rosenbrock function, we don't have real data points, so we'll create a synthetic example
    print("\n--- Chi2/DOF Calculation Example ---")
    
    # Create synthetic linear data for chi2 calculation
    n_points = 10
    x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
    true_slope = 2.0
    true_intercept = 1.0
    y_data_true = true_intercept + true_slope * x_data
    
    # Add some random noise
    np.random.seed(42)
    y_errors = np.ones(n_points) * 0.5  # constant errors
    y_data = y_data_true + np.random.normal(0, 0.3, n_points)
    
    # Create a new minimizer for the linear fit
    minimizer_linear = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    minimizer_linear.SetMaxFunctionCalls(1000000)
    minimizer_linear.SetMaxIterations(10000)
    minimizer_linear.SetTolerance(1e-4)
    minimizer_linear.SetPrecision(1e-10)
    minimizer_linear.SetPrintLevel(0)  # Quiet mode
    
    # Wrap the chi2 function
    def Chi2Wrapper(params):
        return LinearChi2(params, x_data, y_data, y_errors)
    
    f_chi2 = ROOT.Math.Functor(Chi2Wrapper, 2)
    minimizer_linear.SetFunction(f_chi2)
    
    # Set initial parameters for linear fit
    minimizer_linear.SetVariable(0, "intercept", 0.5, 0.1)
    minimizer_linear.SetVariable(1, "slope", 1.5, 0.1)
    
    # Minimize
    ret_linear = minimizer_linear.Minimize()
    
    if ret_linear:
        chi2, dof, chi2_per_dof = CalculateChi2PerDof(minimizer_linear, n_points, 2)
        params = minimizer_linear.X()
        print("Linear fit results:")
        print(f"  Intercept = {params[0]:.4f} (true = {true_intercept})")
        print(f"  Slope = {params[1]:.4f} (true = {true_slope})")
        print(f"  Chi2 = {chi2:.4f}")
        print(f"  Degrees of freedom = {dof}")
        print(f"  Chi2/DOF = {chi2_per_dof:.4f}")
        
        # Interpretation
        if chi2_per_dof < 1.5:
            print("  Good fit: Chi2/DOF < 1.5")
        elif chi2_per_dof < 3.0:
            print("  Reasonable fit: 1.5 < Chi2/DOF < 3.0")
        else:
            print("  Poor fit: Chi2/DOF >= 3.0")
    else:
        print("Linear fit minimization failed!")

    # Real minimum is f(xmin) = 0
    if (ret and minimizer.MinValue() < 1.E-4):
        print("\nMinimizer {} - {} converged to the right minimum!".format(minimizerName, algoName))
    else:
        print("\nMinimizer {} - {} failed to converge !!!".format(minimizerName, algoName))
        raise RuntimeError("NumericalMinimization failed to converge!")


if __name__ == "__main__":
    NumericalMinimization()

In [1]:
import ROOT
import numpy as np
from typing import Callable, List, Tuple, Dict, Any, Optional

def GenericMinimizer(objective_func: Callable,
                    n_dim: int,
                    initial_params: List[float],
                    initial_steps: List[float],
                    param_names: Optional[List[str]] = None,
                    minimizer_name: str = "Minuit2",
                    algorithm_name: str = "",
                    max_iterations: int = 10000,
                    max_function_calls: int = 1000000,
                    tolerance: float = 0.001,
                    print_level: int = 1,
                    fixed_params: Optional[Dict[int, float]] = None,
                    lower_bounds: Optional[Dict[int, float]] = None,
                    upper_bounds: Optional[Dict[int, float]] = None,
                    random_seed: int = -1) -> Dict[str, Any]:
    """
    Generic function for numerical minimization using ROOT's minimizer.
    
    Parameters:
    -----------
    objective_func : Callable
        The objective function to minimize. Should take a list of parameters and return a float.
    n_dim : int
        Number of parameters/dimensions.
    initial_params : List[float]
        Initial values for the parameters.
    initial_steps : List[float]  
        Initial step sizes for the parameters.
    param_names : Optional[List[str]]
        Names for the parameters. If None, will use "param_0", "param_1", etc.
    minimizer_name : str
        Name of the minimizer (e.g., "Minuit2", "Minuit", "GSLMultiMin")
    algorithm_name : str
        Specific algorithm to use (e.g., "Migrad", "Simplex", "BFGS")
    max_iterations : int
        Maximum number of iterations.
    max_function_calls : int
        Maximum number of function calls.
    tolerance : float
        Tolerance for convergence.
    print_level : int
        Print level (0 = quiet, 1 = normal, 2 = verbose).
    fixed_params : Optional[Dict[int, float]]
        Dictionary of parameter indices to fixed values.
    lower_bounds : Optional[Dict[int, float]]
        Dictionary of parameter indices to lower bounds.
    upper_bounds : Optional[Dict[int, float]]
        Dictionary of parameter indices to upper bounds.
    random_seed : int
        Random seed for initial parameter randomization (<0 for fixed initial params).
        
    Returns:
    --------
    Dict[str, Any]
        Dictionary containing minimization results:
        - 'success': bool indicating if minimization was successful
        - 'minimum_value': float, the minimum function value found
        - 'parameters': list of optimized parameters
        - 'parameter_errors': list of parameter errors (if available)
        - 'iterations': number of iterations performed
        - 'function_calls': number of function calls
        - 'status_code': minimizer status code
        - 'minimizer': the minimizer object for further analysis
    """
    
    # Create minimizer
    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizer_name, algorithm_name)
    if not minimizer:
        raise RuntimeError(f"Cannot create minimizer \"{minimizer_name}\". " 
                          "Maybe the required library was not built?")
    
    # Set minimizer parameters
    minimizer.SetMaxFunctionCalls(max_function_calls)
    minimizer.SetMaxIterations(max_iterations)
    minimizer.SetTolerance(tolerance)
    minimizer.SetPrintLevel(print_level)
    
    # Create function wrapper
    func = ROOT.Math.Functor(objective_func, n_dim)
    minimizer.SetFunction(func)
    
    # Handle random initial parameters
    variables = initial_params.copy()
    if random_seed >= 0:
        r = ROOT.TRandom2(random_seed)
        for i in range(n_dim):
            variables[i] = r.Uniform(-10, 10)  # Adjust range as needed
    
    # Set parameters
    if param_names is None:
        param_names = [f"param_{i}" for i in range(n_dim)]
    
    for i in range(n_dim):
        if fixed_params and i in fixed_params:
            # Fixed parameter
            minimizer.SetFixedVariable(i, param_names[i], fixed_params[i])
        elif lower_bounds and upper_bounds and i in lower_bounds and i in upper_bounds:
            # Bounded parameter
            minimizer.SetLimitedVariable(i, param_names[i], variables[i], 
                                       initial_steps[i], lower_bounds[i], upper_bounds[i])
        elif lower_bounds and i in lower_bounds:
            # Lower bound only
            minimizer.SetLowerLimitedVariable(i, param_names[i], variables[i],
                                            initial_steps[i], lower_bounds[i])
        elif upper_bounds and i in upper_bounds:
            # Upper bound only  
            minimizer.SetUpperLimitedVariable(i, param_names[i], variables[i],
                                            initial_steps[i], upper_bounds[i])
        else:
            # Free parameter
            minimizer.SetVariable(i, param_names[i], variables[i], initial_steps[i])
    
    # Perform minimization
    status = minimizer.Minimize()
    
    # Extract results
    result = {
        'success': bool(status),
        'minimum_value': minimizer.MinValue(),
        'parameters': [minimizer.X()[i] for i in range(n_dim)],
        'parameter_errors': [minimizer.Errors()[i] for i in range(n_dim)] if minimizer.Errors() else None,
        'iterations': minimizer.NIterations(),
        'function_calls': minimizer.NCalls(),
        'status_code': status,
        'minimizer': minimizer
    }
    
    return result

def CalculateChi2PerDof(minimizer, n_data_points: int, n_parameters: int) -> Tuple[float, int, float]:
    """
    Calculate chi2 per degree of freedom.
    
    Parameters:
    -----------
    minimizer : ROOT minimizer object
        The minimizer after fitting
    n_data_points : int
        Number of data points
    n_parameters : int
        Number of fitted parameters
        
    Returns:
    --------
    Tuple[float, int, float]
        (chi2, degrees_of_freedom, chi2_per_dof)
    """
    chi2 = minimizer.MinValue()
    dof = n_data_points - n_parameters
    if dof <= 0:
        print("Warning: Degrees of freedom <= 0!")
        return chi2, dof, -1
    
    chi2_per_dof = chi2 / dof
    return chi2, dof, chi2_per_dof


In [2]:

# Example usage functions
def RosenbrockFunction(params: List[float]) -> float:
    """Rosenbrock function for testing minimization."""
    x, y = params[0], params[1]
    return (1 - x)**2 + 100 * (y - x**2)**2

def LinearChi2Function(params: List[float], x_data: np.ndarray, 
                      y_data: np.ndarray, y_errors: np.ndarray) -> float:
    """
    Calculate chi2 for a linear model y = a + b*x.
    
    Parameters:
    -----------
    params : List[float]
        [intercept, slope]
    x_data, y_data, y_errors : np.ndarray
        Data and errors
    """
    a, b = params[0], params[1]
    chi2 = 0.0
    for i in range(len(x_data)):
        y_pred = a + b * x_data[i]
        residual = y_data[i] - y_pred
        chi2 += (residual / y_errors[i])**2
    return chi2

# Example usage
if __name__ == "__main__":
    # Example 1: Minimize Rosenbrock function
    # print("=== Example 1: Rosenbrock Function ===")
    # result1 = GenericMinimizer(
    #     objective_func=RosenbrockFunction,
    #     n_dim=2,
    #     initial_params=[-1.0, 1.2],
    #     initial_steps=[0.01, 0.01],
    #     param_names=["x", "y"],
    #     minimizer_name="Minuit2",
    #     algorithm_name="Migrad",
    #     print_level=1
    # )
    
    # print(f"Success: {result1['success']}")
    # print(f"Minimum value: {result1['minimum_value']:.6f}")
    # print(f"Parameters: {[f'{p:.6f}' for p in result1['parameters']]}")
    # print(f"True minimum at (1, 1) with f=0")
    
    
    # Example 2: Linear fit with chi2
    print("\n=== Example 2: Linear Fit ===")
    
    # Generate synthetic data
    np.random.seed(42)
    n_points = 10
    x_data = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0])
    true_intercept, true_slope = 1.0, 2.0
    y_errors = np.ones(n_points) * 0.5
    y_data = true_intercept + true_slope * x_data + np.random.normal(0, 0.3, n_points)
    
    # Create chi2 function with data bound
    def LinearChi2Wrapper(params):
        return LinearChi2Function(params, x_data, y_data, y_errors)
    
    result2 = GenericMinimizer(
        objective_func=LinearChi2Wrapper,
        n_dim=2,
        initial_params=[0.5, 1.5],
        initial_steps=[0.1, 0.1],
        param_names=["intercept", "slope"],
        minimizer_name="Minuit2",
        algorithm_name="Migrad",
        print_level=1
    )
    
    if result2['success']:
        chi2, dof, chi2_per_dof = CalculateChi2PerDof(
            result2['minimizer'], n_points, 2
        )
        
        print(f"Success: {result2['success']}")
        print(f"Intercept: {result2['parameters'][0]:.4f} (true: {true_intercept})")
        print(f"Slope: {result2['parameters'][1]:.4f} (true: {true_slope})")
        print(f"Chi2: {chi2:.4f}")
        print(f"Chi2/DOF: {chi2_per_dof:.4f}")
        
        # Fit quality assessment
        if chi2_per_dof < 1.5:
            print("Fit quality: Good (χ²/DOF < 1.5)")
        elif chi2_per_dof < 3.0:
            print("Fit quality: Reasonable (1.5 < χ²/DOF < 3.0)")
        else:
            print("Fit quality: Poor (χ²/DOF ≥ 3.0)")
    
    # # Example 3: With bounds
    # print("\n=== Example 3: Bounded Minimization ===")
    # result3 = GenericMinimizer(
    #     objective_func=RosenbrockFunction,
    #     n_dim=2,
    #     initial_params=[-1.0, 1.2],
    #     initial_steps=[0.01, 0.01],
    #     lower_bounds={0: -2.0},  # x >= -2.0
    #     upper_bounds={1: 3.0},   # y <= 3.0
    #     minimizer_name="Minuit2",
    #     print_level=1
    # )
    
    # print(f"Success: {result3['success']}")
    # print(f"Minimum value: {result3['minimum_value']:.6f}")
    # print(f"Parameters: {[f'{p:.6f}' for p in result3['parameters']]}")


=== Example 2: Linear Fit ===
Success: True
Intercept: 1.1458 (true: 1.0)
Slope: 1.9979 (true: 2.0)
Chi2: 1.6923
Chi2/DOF: 0.2115
Fit quality: Good (χ²/DOF < 1.5)
Minuit2Minimizer: Minimize with max-calls 1000000 convergence for edm < 0.001 strategy 1
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = 1.6922668334249189
Edm   = 4.53481364964088537e-22
Nfcn  = 34
intercept	  = 1.1458	 +/-  0.341565
slope	  = 1.99793	 +/-  0.0550482
